# Case Linking — Reference Data Tests

Verifies the static lookup tables that drives caseLinks:
- bronze_case_link_reasons_aria (26 rows, IDs 2–27)
- bronze_case_link_reasons_ccd (17 rows, CLRC001–CLRC017)
- silver_case_link_reason_mappings


In [0]:
########################
# test Setup
########################

state_under_test = "caseLinking"
run_tag_default = "reference_data"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

In [0]:
import os
import sys
import time as perf_time
from datetime import datetime

from pyspark.sql.functions import col
import inspect

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Case_Linking_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"run_tag: {run_tag}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config (state-independent — runs once)
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
env_name = env
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
KeyVault_name = keyvault_name

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

# Storage accounts + spark.conf
curated_storage    = f"ingest{lz_key}curated{env}"
checkpoint_storage = f"ingest{lz_key}xcutting{env}"
raw_storage        = f"ingest{lz_key}raw{env}"
landing_storage    = f"ingest{lz_key}landing{env}"
external_storage   = f"ingest{lz_key}external{env}"

for storage in (curated_storage, checkpoint_storage, raw_storage, landing_storage, external_storage):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_path = f"abfss://bronze@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"

print(f"env={env}  lz_key={lz_key}")

## Load reference tables

In [0]:
bronze_aria_reasons = spark.table("hive_metastore.ariadm_active_appeals.bronze_case_link_reasons_aria")
bronze_ccd_reasons = spark.table("hive_metastore.ariadm_active_appeals.bronze_case_link_reasons_ccd")
silver_reason_mappings = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_reason_mappings")

print(f"bronze_aria_reasons rows:    {bronze_aria_reasons.count()}")
print(f"bronze_ccd_reasons rows:     {bronze_ccd_reasons.count()}")
print(f"silver_reason_mappings rows: {silver_reason_mappings.count()}")

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
t0 = perf_time.time()

#Test bronze_case_link_reasons_ccd

In [0]:
def test_bronze_aria_reasons_content(df):
    expected = {
        2: "Father", 3: "Mother", 4: "Husband", 5: "Wife",
        6: "Son", 7: "Daughter", 8: "Father-in-Law", 9: "Mother-in-Law",
        10: "Brother", 11: "Sister", 12: "Grandfather", 13: "Grandmother",
        14: "Granddaughter", 15: "Grandson", 16: "Grandchild",
        17: "Sister-in-Law", 18: "Brother-in-Law", 19: "Uncle", 20: "Aunt",
        21: "Nephew", 22: "Niece", 23: "Guardian", 24: "Shared Evidence",
        25: "Bail Link", 26: "Other Appeal Pending", 27: "Home Office Request",
    }
    try:
        rows = df.select("ReasonLinkId", "Reason").collect()
        actual = {int(r.ReasonLinkId): r.Reason for r in rows}
    except Exception as e:
        return [TestResult(
            "bronze_case_link_reasons_aria", "FAIL", f"Test crashed: {e}",
            state_under_test, inspect.stack()[0].function)]

    results = []
    fn_name = inspect.stack()[0].function
    for rli in sorted(set(expected) | set(actual)):
        exp = expected.get(rli, "(not expected)")
        act = actual.get(rli, "(missing)")
        status = "PASS" if (rli in expected and rli in actual and expected[rli] == actual[rli]) else "FAIL"
        results.append(TestResult(
            f"bronze_case_link_reasons_aria.{rli}",
            status,
            f"expected: '{exp}' | actual: '{act}'",
            state_under_test, fn_name))
    return results


all_test_results.extend(test_bronze_aria_reasons_content(bronze_aria_reasons))

#Test test_bronze_ccd_reasons

In [0]:
def test_bronze_ccd_reasons_content(df):
    expected = {
        "CLRC001": "Related appeal",
        "CLRC002": "Related proceedings",
        "CLRC003": "Same Party",
        "CLRC004": "Same children",
        "CLRC005": "Familial",
        "CLRC006": "Guardian",
        "CLRC007": "Referred to the same judge",
        "CLRC008": "Shared evidence",
        "CLRC009": "Common circumstance",
        "CLRC010": "Bail",
        "CLRC011": "Findings of fact",
        "CLRC012": "First Tier Agency (FTA) Request",
        "CLRC013": "Point of law",
        "CLRC014": "Other",
        "CLRC015": "Case consolidated",
        "CLRC016": "Progressed as part of this lead case",
        "CLRC017": "Linked for a hearing",
    }
    try:
        rows = df.select("key", "value_en").collect()
        actual = {r.key: r.value_en for r in rows}
    except Exception as e:
        return [TestResult(
            "bronze_case_link_reasons_ccd", "FAIL", f"Test crashed: {e}",
            state_under_test, inspect.stack()[0].function)]

    results = []
    fn_name = inspect.stack()[0].function
    for k in sorted(set(expected) | set(actual)):
        exp = expected.get(k, "(not expected)")
        act = actual.get(k, "(missing)")
        status = "PASS" if (k in expected and k in actual and expected[k] == actual[k]) else "FAIL"
        results.append(TestResult(
            f"bronze_case_link_reasons_ccd.{k}",
            status,
            f"expected: '{exp}' | actual: '{act}'",
            state_under_test, fn_name))
    return results


all_test_results.extend(test_bronze_ccd_reasons_content(bronze_ccd_reasons))

# Test silver_case_link_reason_mappings

In [0]:
def test_silver_reason_mappings_content(df):
    # (ReasonLinkId) -> (expected_key, expected_description)
    expected = {rli: ("CLRC005", None) for rli in range(2, 23)}
    expected[23] = ("CLRC006", None)
    expected[24] = ("CLRC008", None)
    expected[25] = ("CLRC010", None)
    expected[26] = ("CLRC014", "Other Appeal Pending")
    expected[27] = ("CLRC014", "Home Office Request")

    try:
        rows = df.select("ReasonLinkId", "key", "description").collect()
        actual = {
            int(r.ReasonLinkId): (r.key, (r.description if r.description not in ("", None) else None))
            for r in rows
        }
    except Exception as e:
        return [TestResult(
            "silver_case_link_reason_mappings", "FAIL", f"Test crashed: {e}",
            state_under_test, inspect.stack()[0].function)]

    results = []
    fn_name = inspect.stack()[0].function
    for rli in sorted(set(expected) | set(actual)):
        exp = expected.get(rli, "(not expected)")
        act = actual.get(rli, "(missing)")
        status = "PASS" if (rli in expected and rli in actual and expected[rli] == actual[rli]) else "FAIL"
        results.append(TestResult(
            f"silver_case_link_reason_mappings.{rli}",
            status,
            f"expected: {exp} | actual: {act}",
            state_under_test, fn_name))
    return results


all_test_results.extend(test_silver_reason_mappings_content(silver_reason_mappings))

In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

## Write reports + audit

In [0]:
# Sort fails to the top, then alphabetical by test_field within each status bucket.
status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
all_test_results.sort(key=lambda r: (status_order.get(str(r.status).upper(), 4), str(r.test_field)))

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path  = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

# try:
#     xlsx_path = write_xlsx(all_test_results, state_under_test, folder, timestamp)
#     print(f"XLSX : {xlsx_path}")
# except Exception as e:
#     print(f"write_xlsx FAILED: {e}")

# try:
#     html_path = write_html(all_test_results, state_under_test, folder, timestamp, counts=counts)
#     print(f"HTML : {html_path}")
# except Exception as e:
#     print(f"write_html FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Case_Linking_Tests",
        run_tag=run_tag,
        run_status="COMPLETED",
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
except Exception as e:
    print(f"audit write FAILED: {e}")